[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/adan-rs/amd/blob/main/notebooks/18_RLM_extensiones.ipynb)

# Análisis de regresión: extensiones del modelo

El notebook anterior (16_RLM) mostró el caso "estándar" de la regresión lineal múltiple: variables independientes cuantitativas, relación lineal y efectos que se suman entre sí. En la práctica de negocios, con frecuencia se necesita extender ese modelo básico en tres direcciones:

- **Variables categóricas**: ¿cómo incluyo el género del cliente, la sucursal o el día de la semana en un modelo que solo admite números?
- **Interacción**: ¿el efecto de una variable sobre la variable dependiente cambia según el valor de otra variable? (por ejemplo, ¿el efecto de la publicidad en ventas es distinto en tienda física que en línea?)
- **Transformación de variables**: ¿qué hago cuando la relación entre las variables no es lineal, o cuando me interesa interpretar el efecto en términos porcentuales (elasticidad) en vez de unidades absolutas?

Este notebook cubre las tres extensiones. Para variables categóricas usaremos datos de viajes en Uber/taxi en Monterrey; para interacción y transformación retomamos el caso de precios de vivienda del notebook 16.

## 1. Variables categóricas

*¿Para qué se utiliza?*
Permite incluir predictores cualitativos (género, sucursal, día de la semana, tipo de producto) dentro de un modelo de regresión, que por diseño solo admite variables numéricas.

Ejemplos de uso en negocios:
- ¿El gasto promedio es distinto entre clientes hombres y mujeres, controlando por el gasto en publicidad?
- ¿La duración de un viaje cambia según el día de la semana?
- ¿El desempeño de ventas es distinto entre sucursales, una vez controlado el tamaño del equipo?

*Variables consideradas*
Una variable categórica nominal u ordinal con k categorías se representa mediante k − 1 variables dicotómicas (dummies), cada una codificada como 1 si la observación pertenece a esa categoría y 0 en caso contrario. Una de las categorías se deja fuera como referencia o categoría base; todas las comparaciones se interpretan respecto a ella.

*Interpretación de los coeficientes*

Supón que estimas el siguiente modelo:

Ventas = β₀ + β₁ · Publicidad + β₂ · Genero_femenino + ε

Y el resultado de la regresión es β₀ = 120, β₁ = 5.2, β₂ = 18.7. La interpretación es:
- Por cada unidad adicional de gasto en publicidad, las ventas aumentan en promedio 5.2 unidades (β₁).
- Si el cliente es mujer (Genero_femenino = 1), las ventas aumentan en promedio 18.7 unidades respecto a los hombres (Genero_femenino = 0), manteniendo constante el gasto en publicidad (β₂).

*Cuidado clave: la trampa de las variables dummy*
Si se incluyen las k dummies de una variable categórica junto con la constante del modelo, se genera multicolinealidad perfecta (las k dummies siempre suman 1). Por eso siempre se omite una categoría como referencia — en pandas esto se logra con `drop_first=True` en `pd.get_dummies()`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm

El archivo *ubermty* contiene información de viajes en taxi o Uber en la ciudad de Monterrey de junio de 2016 a agosto de 2017. Crearemos un modelo para estimar la duración de cada viaje en función de la distancia y el día de la semana.

In [ ]:
variables = ['id', 'Dia', 'DuracionMinutos', 'DistanciaKm']

# Cargar los datos
df = pd.read_excel('https://github.com/adan-rs/amd/raw/main/data/ubermty.xlsx', usecols=variables)

In [ ]:
df.info()

In [ ]:
df.describe().T

Filtramos valores atípicos en la duración del viaje usando el criterio del rango intercuartílico.

In [ ]:
Q1 = df['DuracionMinutos'].quantile(0.25)
Q3 = df['DuracionMinutos'].quantile(0.75)
IQR = Q3-Q1
lim_inf = Q1-1.5*IQR
lim_sup = Q3+1.5*IQR
df = df[(df['DuracionMinutos']>lim_inf)&(df['DuracionMinutos']<lim_sup)]

In [ ]:
sns.boxplot(x='Dia', y='DuracionMinutos', data=df);

**Creación de variables dicotómicas**: la función `get_dummies` convierte una variable categórica (con k categorías) en k variables dicotómicas. Entre otros parámetros se puede establecer:
- `prefix`: agrega un prefijo a los nombres de las columnas.
- `dtype`: de manera predeterminada está en *booleano*, se puede cambiar a *int*.
- `drop_first`: quita la columna de la primera categoría (la que queda como referencia).

In [ ]:
# Crear variables dicotómicas (se omite 'domingo', que queda como categoría de referencia)
df = pd.get_dummies(df, columns=['Dia'], prefix='dia',
                    dtype=int, drop_first=True)
df.head(3)

In [ ]:
X = df[['DistanciaKm', 'dia_lunes', 'dia_martes', 'dia_miercoles',
        'dia_jueves', 'dia_viernes', 'dia_sabado']]
X = sm.add_constant(X)
y = df['DuracionMinutos']

model = sm.OLS(y, X).fit()
print(model.summary())

**Ejemplo de reporte de resultados**:
>"Se estimó un modelo de regresión múltiple para explicar la duración de un viaje (en minutos) en función de la distancia recorrida y el día de la semana, usando el domingo como categoría de referencia. El modelo fue significativo (F(7, 8732) = 1537, p < 0.001) y explicó el 55.2% de la varianza (R² = 0.552). Por cada kilómetro adicional, la duración del viaje aumenta en promedio 1.29 minutos (p < 0.001). Todos los días de la semana mostraron una duración significativamente mayor que el domingo, con el efecto más alto el viernes (+2.02 minutos) y el más bajo el sábado (+0.72 minutos), consistente con patrones de tráfico más intensos entre semana."

## 2. Interacción

*¿Para qué se utiliza?*
Permite modelar el caso en que el efecto de una variable independiente sobre la dependiente no es constante, sino que depende del valor de otra variable. Sin un término de interacción, el modelo asume que el efecto de cada variable es el mismo para todas las observaciones.

Ejemplos de uso en negocios:
- ¿El efecto de la inversión en publicidad sobre las ventas es distinto en temporada alta que en temporada baja?
- ¿El impacto de la antigüedad de un empleado en su productividad depende del área en la que trabaja?
- ¿El efecto del tamaño de construcción sobre el precio de una propiedad es el mismo para casas que para departamentos?

Retomamos el caso de precios de vivienda del notebook 16.

*Cuidado clave*: cuando el modelo incluye un término de interacción, el coeficiente de la variable principal ya no representa su efecto general, sino su efecto únicamente cuando la variable con la que interactúa vale cero. Interpretar ese coeficiente de forma aislada, ignorando la interacción, es un error común.

In [ ]:
df = pd.read_excel('https://github.com/adan-rs/amd/raw/main/data/casas.xlsx',
                   usecols=['preciomillones', 'construccion', 'tipo'])
df.head()

In [ ]:
# Modelo sin interacción
y = df['preciomillones']
X = sm.add_constant(df[['construccion', 'tipo']])

modelo = sm.OLS(y, X).fit()
modelo.summary()

In [ ]:
sns.scatterplot(data=df, x='construccion', y='preciomillones', hue='tipo');

Para capturar si el efecto de la construcción sobre el precio depende del tipo de propiedad, agregamos un término de interacción (el producto de ambas variables).

In [ ]:
# Calcular el término de interacción
df['construccion_tipo'] = df['construccion'] * df['tipo']

y = df['preciomillones']
X = sm.add_constant(df[['construccion', 'tipo', 'construccion_tipo']])

resultado = sm.OLS(y, X).fit()
resultado.summary()

**Interpretación**: el coeficiente del término de interacción es negativo y significativo, lo que indica que el efecto de la construcción sobre el precio es menor para los departamentos (tipo = 1) que para las casas (tipo = 0): cada metro cuadrado adicional de construcción aumenta menos el precio de los departamentos que el de las casas.

**Ejemplo de reporte de resultados**:
>"Se estimó un modelo de regresión con interacción para evaluar si el efecto de la superficie de construcción sobre el precio de venta depende del tipo de propiedad (casa o departamento). El modelo fue significativo (R² = 0.898) y el término de interacción resultó estadísticamente significativo (β = −0.020, p < 0.001). Por cada metro cuadrado adicional de construcción, el precio aumenta en promedio 0.055 millones de pesos en las casas, pero solo 0.035 millones de pesos en los departamentos. Esto indica que el metro cuadrado adicional se valora más en casas que en departamentos, y que ignorar la interacción habría subestimado esta diferencia."

In [ ]:
sns.lmplot(x='construccion', y='preciomillones', data=df, hue='tipo', markers=['o', 's']);

In [ ]:
# Pronóstico: construccion=500, tipo=1 (departamento), construccion_tipo=500
nuevos_valores = [1, 500, 1, 500]
resultado.predict(nuevos_valores)

## 3. Transformación de variables

*¿Para qué se utiliza?*
Cuando la relación entre dos variables no es lineal, o cuando interesa interpretar el efecto en términos relativos (porcentuales) en lugar de unidades absolutas, es común transformar las variables antes de estimar el modelo. La transformación logarítmica es la más utilizada en negocios porque permite interpretar los coeficientes como elasticidades.

Ejemplos de uso en negocios:
- ¿En qué porcentaje cambia la demanda de un producto ante un cambio de 1% en su precio? (elasticidad precio-demanda)
- ¿En qué porcentaje aumenta el precio de una vivienda por cada 1% adicional de superficie construida?
- ¿En qué porcentaje varía el gasto de un hogar ante un cambio de 1% en su ingreso?

Continuamos con el caso de precios de vivienda.

*Interpretación*: cuando tanto la variable dependiente como la independiente se transforman con logaritmo natural (modelo log-log), el coeficiente estimado se interpreta directamente como una elasticidad: el cambio porcentual en Y ante un cambio porcentual de 1% en X.

$$\ln(Y) = \beta_0 + \beta_1 \ln(X) + \varepsilon \qquad \beta_1 = \frac{d(\ln Y)}{d(\ln X)} = \text{Elasticidad}$$

In [ ]:
sns.scatterplot(data=df, x='construccion', y='preciomillones');

In [ ]:
df['log_construccion'] = np.log(df['construccion'])
df['log_precio'] = np.log(df['preciomillones'])

In [ ]:
sns.scatterplot(data=df, x='log_construccion', y='log_precio');

In [ ]:
X = sm.add_constant(df['log_construccion'])
y = df['log_precio']

model = sm.OLS(y, X).fit()
print(model.summary())

**Ejemplo de reporte de resultados**:
>"Se estimó un modelo log-log para evaluar la elasticidad del precio de una vivienda respecto a su superficie de construcción. El modelo explicó el 85.5% de la varianza (R² = 0.855) y el coeficiente estimado fue de 1.08 (p < 0.001), lo que indica una elasticidad prácticamente unitaria: por cada incremento de 1% en la superficie construida, el precio de la vivienda aumenta en promedio 1.08%."

## Ejercicio

El archivo `enigh2025.xlsx` contiene el ingreso corriente (`ing_cor`) y el gasto monetario (`gasto_mon`) de una muestra de hogares.

1. Estima un modelo lineal simple de `gasto_mon` en función de `ing_cor` y observa el ajuste (R²) y el diagrama de dispersión de las variables.
2. Transforma ambas variables con logaritmo natural y vuelve a estimar el modelo.
3. Compara el ajuste de ambos modelos e interpreta el coeficiente del modelo log-log como una elasticidad. ¿Por cada 1% adicional de ingreso, en qué porcentaje aumenta el gasto de los hogares?

In [ ]:
df = pd.read_excel('https://github.com/adan-rs/amd/raw/main/data/enigh2025.xlsx',
                   usecols=['ing_cor', 'gasto_mon'])